In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────
SHAPE_LOG       = r"C:/ClaudeCode/research/01. regime_identification/input/shape_log.csv"
ENRICHED_LOG    = r"C:/ClaudeCode/research/03. validation_analysis/shape_log_enriched.csv"
STOCK           = r"C:/ClaudeCode/Raw Data/Stock and Production/FCPO Stock 3Y.xlsx"
PRODUCTION      = r"C:/ClaudeCode/Raw Data/Stock and Production/MPOB Production 3Y.xlsx"
EXPORT          = r"C:/ClaudeCode/Raw Data/Stock and Production/MPOB Export 3Y.xlsx"
USDMYR          = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/FX_IDC_USDMYR_1D_1227e.xlsx"
PALMSOY         = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/MYX_DLY_FCPO1!_2_CBOT_DL_ZL1!_1D_6.xlsx"
CRUDE           = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/NYMEX_DL_CL1!_1D_84001.xlsx"
ENSO_ASCII      = r"C:/ClaudeCode/Raw Data/ENSO/oni.ascii.txt"
LOG_FILE        = r"C:/ClaudeCode/research/03. validation_analysis/further_validation_log.txt"

# ── shape name mapping ─────────────────────────────────────────
# Internal code uses 0.0/0.1/0.2/1/2
# Display layer converts to readable names
SHAPE_NAMES = {
    '0.0': 'SB  (Steep Backwardation)',
    '0.1': 'MB  (Mild Backwardation)',
    '0.2': 'TB  (Transitional Backwardation)',
    '1':   'C   (Contango)',
    '2':   'F   (Flat/Back-Inversion)',
}
SHAPE_SHORT = {
    '0.0': 'SB', '0.1': 'MB',
    '0.2': 'TB', '1': 'C', '2': 'F',
}

def display_shape(code):
    return SHAPE_NAMES.get(str(code), str(code))

def display_short(code):
    return SHAPE_SHORT.get(str(code), str(code))

# ── log writer ─────────────────────────────────────────────────
def write_log(study_id, content):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    header = f"""
{"="*70}
{study_id}
Run: {timestamp}
{"="*70}
"""
    with open(LOG_FILE, 'a') as f:
        f.write(header + content + "\n")
    print(f"[LOG] Written — {study_id}")

# ── train/test split (locked, same as all prior work) ──────────
TRAIN_END  = '2024-12-31'
TEST_START = '2025-01-01'

print("Setup complete.")
print("Shape name mapping:")
for k, v in SHAPE_NAMES.items():
    print(f"  {k} -> {v}")

In [ ]:
# ── load shape log (enriched version with duration + prior) ───
df_enriched = pd.read_csv(
    ENRICHED_LOG,
    dtype={'shape': str, 'prior_shape': str},
    parse_dates=['date'])
df_enriched = df_enriched[
    df_enriched['date'] >= '2017-01-01'
].sort_values('date').reset_index(drop=True)

# ── load MPOB ──────────────────────────────────────────────────
def load_mpob(path, col_name):
    df = pd.read_excel(path, parse_dates=True)
    date_col = df.columns[0]
    val_col  = df.columns[1]
    df = df.rename(columns={date_col: 'date', val_col: col_name})
    df['date'] = pd.to_datetime(df['date'])
    return df[['date', col_name]].sort_values('date')

stock = load_mpob(STOCK,      'stock_raw')
prod  = load_mpob(PRODUCTION, 'production_raw')
exp   = load_mpob(EXPORT,     'export_raw')

# ── load macro ─────────────────────────────────────────────────
def load_macro(path, col_name):
    df = pd.read_excel(path, parse_dates=True)
    date_col = df.columns[0]
    val_col  = df.columns[1]
    df = df.rename(columns={date_col: 'date', val_col: col_name})
    df['date'] = pd.to_datetime(df['date'])
    return df[['date', col_name]].sort_values('date')

usdmyr  = load_macro(USDMYR,   'usd_myr')
palmsoy = load_macro(PALMSOY,  'palm_soy')
crude   = load_macro(CRUDE,    'crude_oil')

# ── load ENSO ──────────────────────────────────────────────────
enso = pd.read_csv(ENSO_ASCII, sep=r'\s+',
    names=['year','jan','feb','mar','apr','may',
           'jun','jul','aug','sep','oct','nov','dec'])
enso_long = enso.melt(
    id_vars='year', var_name='month', value_name='oni')
month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,
             'jun':6,'jul':7,'aug':8,'sep':9,
             'oct':10,'nov':11,'dec':12}
enso_long['month'] = enso_long['month'].map(month_map)
enso_long['date']  = pd.to_datetime(
    enso_long[['year','month']].assign(day=1))
enso_long = enso_long[['date','oni']].dropna()

# ── merge everything onto enriched shape log ───────────────────
df = df_enriched.copy()

# MPOB monthly — merge as of (forward fill)
for src, col in [(stock,'stock_raw'),
                  (prod, 'production_raw'),
                  (exp,  'export_raw')]:
    df = pd.merge_asof(
        df.sort_values('date'),
        src.sort_values('date'),
        on='date', direction='backward')

# ENSO monthly
df = pd.merge_asof(
    df.sort_values('date'),
    enso_long.sort_values('date'),
    on='date', direction='backward')

# Macro daily
for src, col in [(usdmyr,'usd_myr'),
                  (palmsoy,'palm_soy'),
                  (crude,  'crude_oil')]:
    df = pd.merge_asof(
        df.sort_values('date'),
        src.sort_values('date'),
        on='date', direction='backward')

# ── derived variables ──────────────────────────────────────────
df['stock_pct']        = df['stock_raw'].rank(pct=True)
df['prod_mom_1m']      = df['production_raw'].pct_change(21)*100
df['prod_mom_3m']      = df['production_raw'].pct_change(63)*100
df['prod_yoy']         = df['production_raw'].pct_change(252)*100
df['export_yoy']       = df['export_raw'].pct_change(252)*100
df['usd_myr_chg_4w']   = df['usd_myr'].pct_change(20)*100
df['palm_soy_chg_4w']  = df['palm_soy'].pct_change(20)*100
df['crude_chg_4w']     = df['crude_oil'].pct_change(20)*100
df['month']            = df['date'].dt.month

# ── encode prior shape ─────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['prior_shape_enc'] = le.fit_transform(
    df['prior_shape'].fillna('unknown'))

# ── stock x production 2D interaction ─────────────────────────
df['stock_state'] = np.where(
    df['stock_pct'] >= df['stock_pct'].median(),
    'High', 'Low')
df['prod_state'] = np.where(
    df['prod_mom_3m'] >= 0, 'Rising', 'Falling')
df['stock_prod_quad'] = df['stock_state'] + '_' + df['prod_state']

quad_map = {
    'High_Falling': 3,  # Abundant but Tightening (WATCH)
    'Low_Rising':   2,  # Tight but Recovering (WATCH)
    'Low_Falling':  1,  # Tight and Worsening
    'High_Rising':  0,  # Abundant and Building (most stable)
}
df['stock_prod_interaction'] = df['stock_prod_quad'].map(
    quad_map).fillna(0)

print(f"Panel built: {len(df)} rows")
print(f"Date range: {df['date'].min().date()} to "
      f"{df['date'].max().date()}")
print(f"\nMissing values in key columns:")
key_cols = ['stock_pct','prod_mom_3m','oni',
            'usd_myr_chg_4w','palm_soy_chg_4w',
            'prior_shape_enc','days_in_shape']
print(df[key_cols].isnull().sum())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,
    confusion_matrix, accuracy_score)
from sklearn.model_selection import (cross_val_score,
    StratifiedKFold)
from sklearn.feature_selection import RFECV
import matplotlib
matplotlib.use('Agg')

# ── full feature set (all candidates) ─────────────────────────
ALL_FEATURES = {
    # MPOB fundamentals
    'stock_pct':           'Stock Percentile',
    'prod_mom_1m':         'Production Mom 1m',
    'prod_mom_3m':         'Production Mom 3m',
    'prod_yoy':            'Production YoY',
    'export_yoy':          'Export YoY',

    # Macro
    'oni':                 'ENSO ONI',
    'usd_myr_chg_4w':      'USD/MYR Change 4w',
    'palm_soy_chg_4w':     'Palm-Soy Spread 4w',
    'crude_chg_4w':        'Crude Oil Change 4w',

    # Curve history (new signals from validation)
    'days_in_shape':       'Days in Current Shape',
    'prior_shape_enc':     'Prior Shape',
    'month':               'Calendar Month',

    # 2D interaction
    'stock_prod_interaction': 'Stock x Production Quad',
}

# ── backup feature set (current 74.8% model) ──────────────────
BACKUP_FEATURES = [
    'stock_pct', 'prod_mom_1m', 'prod_mom_3m', 'month'
]

feature_cols = [f for f in ALL_FEATURES.keys()
                if f in df.columns]
print(f"Features available: {len(feature_cols)}")
print(f"Features missing: "
      f"{[f for f in ALL_FEATURES if f not in df.columns]}")

# ── train/test split ───────────────────────────────────────────
df_model = df.dropna(subset=feature_cols+['shape']).copy()
df_train = df_model[df_model['date'] <= TRAIN_END]
df_test  = df_model[df_model['date'] >= TEST_START]

X_train = df_train[feature_cols].values
y_train = df_train['shape'].values
X_test  = df_test[feature_cols].values
y_test  = df_test['shape'].values

print(f"\nTrain: {len(df_train)} rows "
      f"({df_train['date'].min().date()} to "
      f"{df_train['date'].max().date()})")
print(f"Test:  {len(df_test)} rows "
      f"({df_test['date'].min().date()} to "
      f"{df_test['date'].max().date()})")
print(f"\nTrain shape distribution:")
for s, n in pd.Series(y_train).value_counts().items():
    print(f"  {display_short(s)}: {n}")

In [ ]:
# ── build backup model on existing feature set ────────────────
# This is the reference. We never replace this.
X_train_bk = df_train[BACKUP_FEATURES].values
X_test_bk  = df_test[BACKUP_FEATURES].values

backup_model = RandomForestClassifier(
    n_estimators=200, max_depth=5,
    min_samples_leaf=10, random_state=42,
    class_weight='balanced')
backup_model.fit(X_train_bk, y_train)

backup_pred   = backup_model.predict(X_test_bk)
backup_acc    = accuracy_score(y_test, backup_pred)*100
backup_cv     = cross_val_score(
    backup_model, X_train_bk, y_train,
    cv=StratifiedKFold(5), scoring='accuracy').mean()*100

print("BACKUP MODEL (74.8% reference)")
print("="*50)
print(f"CV accuracy:   {backup_cv:.1f}%")
print(f"Test accuracy: {backup_acc:.1f}%")
print("\nPer-shape F1 (backup):")
report = classification_report(
    y_test, backup_pred, output_dict=True)
for s in sorted(df['shape'].unique()):
    if s in report:
        f1  = report[s]['f1-score']*100
        prec = report[s]['precision']*100
        rec  = report[s]['recall']*100
        print(f"  {display_short(s)}: "
              f"F1={f1:.0f}%  "
              f"Prec={prec:.0f}%  "
              f"Rec={rec:.0f}%")

output_lines = [
    "BACKUP MODEL (74.8% reference — DO NOT REPLACE)",
    f"CV accuracy:   {backup_cv:.1f}%",
    f"Test accuracy: {backup_acc:.1f}%",
    "\nPer-shape results:"
]
for s in sorted(df['shape'].unique()):
    if s in report:
        output_lines.append(
            f"  {display_shape(s)}: "
            f"F1={report[s]['f1-score']*100:.0f}%  "
            f"Prec={report[s]['precision']*100:.0f}%  "
            f"Rec={report[s]['recall']*100:.0f}%")

write_log("FURTHER STUDY 1 — BACKUP MODEL REFERENCE",
          "\n".join(output_lines))

In [ ]:
# ── new model with full variable set ──────────────────────────
new_model = RandomForestClassifier(
    n_estimators=300, max_depth=6,
    min_samples_leaf=8, random_state=42,
    class_weight='balanced')
new_model.fit(X_train, y_train)

new_pred = new_model.predict(X_test)
new_acc  = accuracy_score(y_test, new_pred)*100
new_cv   = cross_val_score(
    new_model, X_train, y_train,
    cv=StratifiedKFold(5),
    scoring='accuracy').mean()*100

print("NEW MODEL (full variable set)")
print("="*50)
print(f"CV accuracy:   {new_cv:.1f}%")
print(f"Test accuracy: {new_acc:.1f}%")
print(f"vs backup:     {new_acc-backup_acc:+.1f}pp")

print("\nPer-shape F1 comparison (backup vs new):")
new_report = classification_report(
    y_test, new_pred, output_dict=True)
output_lines = [
    "NEW MODEL — FULL VARIABLE SET",
    f"CV accuracy:   {new_cv:.1f}%",
    f"Test accuracy: {new_acc:.1f}%",
    f"vs backup:     {new_acc-backup_acc:+.1f}pp",
    "\nPer-shape comparison:"
]
for s in sorted(df['shape'].unique()):
    bk_f1  = report.get(s,{}).get('f1-score',0)*100
    new_f1 = new_report.get(s,{}).get('f1-score',0)*100
    diff   = new_f1 - bk_f1
    flag   = " ***" if diff > 5 else (" !" if diff < -5 else "")
    line   = (f"  {display_shape(s)}: "
              f"backup F1={bk_f1:.0f}%  "
              f"new F1={new_f1:.0f}%  "
              f"diff={diff:+.0f}pp{flag}")
    print(line)
    output_lines.append(line)

# Feature importance
print("\nFeature importance (new model):")
fi = sorted(zip(feature_cols,
                new_model.feature_importances_),
            key=lambda x: -x[1])
output_lines.append("\nFeature importance:")
for feat, imp in fi:
    line = (f"  {ALL_FEATURES.get(feat,feat):<35}"
            f"{imp:.3f}")
    print(line)
    output_lines.append(line)

write_log("FURTHER STUDY 1 — NEW MODEL (FULL VARIABLES)",
          "\n".join(output_lines))

In [ ]:
# ── recursive feature elimination ─────────────────────────────
# Finds the optimal subset of variables that maximises
# cross-validated accuracy — removes variables that
# add noise rather than signal

from sklearn.feature_selection import RFECV

print("Running variable optimization...")
print("(This may take a few minutes)")

selector = RFECV(
    estimator=RandomForestClassifier(
        n_estimators=100, max_depth=5,
        min_samples_leaf=10, random_state=42,
        class_weight='balanced'),
    step=1,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    min_features_to_select=3,
    n_jobs=-1)

selector.fit(X_train, y_train)

optimal_features = [feature_cols[i]
                    for i in range(len(feature_cols))
                    if selector.support_[i]]

print(f"\nOptimal feature count: {selector.n_features_}")
print(f"Best CV score: {selector.cv_results_['mean_test_score'].max()*100:.1f}%")
print(f"\nSelected features:")
for f in optimal_features:
    print(f"  {ALL_FEATURES.get(f,f)}")

print(f"\nRemoved features:")
removed = [f for f in feature_cols
           if f not in optimal_features]
for f in removed:
    print(f"  {ALL_FEATURES.get(f,f)} — added noise, removed")

output_lines = [
    "VARIABLE OPTIMIZATION — RFECV RESULTS",
    f"Optimal feature count: {selector.n_features_}",
    f"Best CV score: "
    f"{selector.cv_results_['mean_test_score'].max()*100:.1f}%",
    "\nSelected features:"
]
for f in optimal_features:
    output_lines.append(f"  {ALL_FEATURES.get(f,f)}")
output_lines.append("\nRemoved features:")
for f in removed:
    output_lines.append(
        f"  {ALL_FEATURES.get(f,f)} — added noise, removed")

write_log("FURTHER STUDY 1 — VARIABLE OPTIMIZATION",
          "\n".join(output_lines))

In [ ]:
# ── build final optimized model ───────────────────────────────
X_train_opt = df_train[optimal_features].values
X_test_opt  = df_test[optimal_features].values

opt_model = RandomForestClassifier(
    n_estimators=300, max_depth=6,
    min_samples_leaf=8, random_state=42,
    class_weight='balanced')
opt_model.fit(X_train_opt, y_train)

opt_pred = opt_model.predict(X_test_opt)
opt_acc  = accuracy_score(y_test, opt_pred)*100
opt_cv   = cross_val_score(
    opt_model, X_train_opt, y_train,
    cv=StratifiedKFold(5),
    scoring='accuracy').mean()*100

print("OPTIMIZED MODEL (best variable subset)")
print("="*60)
print(f"\n{'Model':<20} {'CV Acc':>8} {'Test Acc':>10} "
      f"{'vs Backup':>10}")
print("-"*50)
print(f"{'Backup (74.8%)':<20} {backup_cv:>7.1f}% "
      f"{backup_acc:>9.1f}% {'—':>10}")
print(f"{'Full variables':<20} {new_cv:>7.1f}% "
      f"{new_acc:>9.1f}% {new_acc-backup_acc:>+9.1f}pp")
print(f"{'Optimized subset':<20} {opt_cv:>7.1f}% "
      f"{opt_acc:>9.1f}% {opt_acc-backup_acc:>+9.1f}pp")

print("\nPer-shape F1 — all three models:")
opt_report = classification_report(
    y_test, opt_pred, output_dict=True)

output_lines = [
    "OPTIMIZED MODEL — FINAL COMPARISON",
    f"{'Model':<20} {'CV':>8} {'Test':>8} {'vs Backup':>10}",
    f"Backup:           {backup_cv:.1f}%   "
    f"{backup_acc:.1f}%   —",
    f"Full variables:   {new_cv:.1f}%   "
    f"{new_acc:.1f}%   {new_acc-backup_acc:+.1f}pp",
    f"Optimized subset: {opt_cv:.1f}%   "
    f"{opt_acc:.1f}%   {opt_acc-backup_acc:+.1f}pp",
    "\nPer-shape F1:"
]

for s in sorted(df['shape'].unique()):
    bk  = report.get(s,{}).get('f1-score',0)*100
    new = new_report.get(s,{}).get('f1-score',0)*100
    opt = opt_report.get(s,{}).get('f1-score',0)*100
    line = (f"  {display_shape(s)}: "
            f"backup={bk:.0f}%  "
            f"full={new:.0f}%  "
            f"optimized={opt:.0f}%")
    print(line)
    output_lines.append(line)

# Verdict
best_acc = max(backup_acc, new_acc, opt_acc)
best_name = (
    "Optimized" if opt_acc == best_acc else
    "Full variables" if new_acc == best_acc
    else "Backup")

output_lines.append(
    f"\nVERDICT: Best model = {best_name} "
    f"at {best_acc:.1f}% test accuracy")
print(f"\nVERDICT: Best model = {best_name} "
      f"at {best_acc:.1f}% test accuracy")

write_log("FURTHER STUDY 1 — OPTIMIZED MODEL FINAL",
          "\n".join(output_lines))
print("\n[NOTEBOOK 1 COMPLETE]")